In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from torch.utils.data import DataLoader, random_split
from sklearn.model_selection import train_test_split
import torch.utils.data as data
import torch.nn as nn
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torchsummary import summary
from sklearn.preprocessing import StandardScaler

Loading the dataset and visualising to know the number of features and labels(targets) that are present in the dataset.

In [3]:
# Loading the  given dataset
data_set = pd.read_csv(r"/content/Assignment1_regression_dataset.csv")

# Display the data
print(data_set)


FileNotFoundError: [Errno 2] No such file or directory: '/content/Assignment1_regression_dataset.csv'

Seperating the data into inputs and targets.

In [ ]:
# converting the data into labelled data containg input and ground truth

# input data (features- first 20 colums(all rows except last 2 colums))
X = data_set.iloc[:,:-2].values

# ground truth (labels) corresponding to the inputs(all rows last 2 columns)
Y = data_set.iloc[:,-2:].values



Standardizing the data.

In [ ]:
# Standardize the features for better training stability
scaler = StandardScaler()
scaler_Y = StandardScaler()
X = scaler.fit_transform(X)
Y = scaler_Y.fit_transform(Y)

In [ ]:
# Task-1 : splitting the data

In [ ]:
def split_data(X, y, training_dr=0.7, validation_dr=0.15, testing_dr=0.15, batch_size=30):

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32)
    dataset = data.TensorDataset(X_tensor, y_tensor)

    # Calculate sizes
    td_size = int(training_dr * len(dataset))
    vd_size = int(validation_dr * len(dataset))
    testd_size = len(dataset) - (td_size + vd_size)

    # Splitting  dataset
    train_set, val_set, test_set = data.random_split(dataset, [td_size, vd_size, testd_size])

    # Creating different data sets
    training_data = data.DataLoader(train_set, batch_size=batch_size, shuffle=True,num_workers=0)
    validation_data = data.DataLoader(val_set, batch_size=batch_size, shuffle=True,num_workers=0)
    testing_data = data.DataLoader(test_set, batch_size=batch_size, shuffle=True,num_workers=0)

    return training_data, validation_data, testing_data


In [ ]:
# generating datasets
batch_size=30
training_data, validation_data, testing_data = split_data(X, Y)

In [ ]:
# Task 2: Designing a Fully Connected Neural Network for Regression

In [ ]:
class RegressionModel(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim):
        super(RegressionModel, self).__init__()
        self.l1 = nn.Linear(input_dim,hidden_layers[0])
        self.relu1 = nn.ReLU()
        self.l2 = nn.Linear(hidden_layers[0],hidden_layers[1])
        self.relu2 = nn.ReLU()
        self.l3 = nn.Linear(hidden_layers[1], hidden_layers[2])
        self.relu3 = nn.ReLU()
        self.l4 = nn.Linear(hidden_layers[2], output_dim)

    def forward(self, x):
        x = self.l1(x)
        x = self.relu1(x)
        x = self.l2(x)
        x = self.relu2(x)
        x = self.l3(x)
        x = self.relu3(x)
        x = self.l4(x)

        return x


I have not used any activation at the end of the nureal network which is output layer. Because nueral network for regression doesn't need any classification (like sigmoid can use for binary classification and softmax for multi class classification).
        Here we just having predictions and applying an activation at the output may alter the predictions.

In [ ]:
# Define Model Parameters
input_dim = X.shape[1]
hidden_layers = [64, 32, 16]
output_dim = Y.shape[1]

# Initialize Model
model = RegressionModel(input_dim, hidden_layers, output_dim)


Input layer contains 20 inputs as we are having 20 features and defined neural network with 3 hidden layers containg 64,32,16 pereceptrons respectively, finally an output layer with two perceptrons for two targets y_1,y_2

In [ ]:
# Task 3: Print Model Summary
print(model)

In [ ]:
# Print model summary
summary(model, input_size=(batch_size, input_dim))




*   Trainable parameters in hidden layer1(weights + biases) FC layer = 20*64 + 64 = 1344.
*   Relu is an logical function doesn't have any trainable parameters

*   similarly in hidden layer 2  = 64*32 + 32 = 2080

*   hidden layer 3 = 20*16+16 = 528

*   Output layer = 16*2+2 = 34

*   Total output parameters = 3986












In [ ]:
# Task 4: Define Loss Function
loss_fn = nn.MSELoss()


Mean Square is an appropriate loss function which can used to calculate how far are the predictions from the ground truth points. Which tell us that amount of deviation.

In [ ]:
# Task 5: Training

In [ ]:
def train_model(model, training_data, validation_data, epochs=30, lr=0.01):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    writer = SummaryWriter()
    training_losses = [] #stores the training loss occured in every epoch
    validation_losses = [] #stores the validation loss occured in every epoch

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x_batch, y_batch in training_data:
            optimizer.zero_grad()
            predictions = model(x_batch)
            loss = loss_fn(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(training_data)
        training_losses.append(train_loss)

        # Validation Phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in validation_data:
                predictions = model(x_batch)
                val_loss += loss_fn(predictions, y_batch).item()
        val_loss /= len(validation_data)
        validation_losses.append(val_loss)

        scheduler.step(val_loss)
        writer.add_scalars('Loss', {'Train': train_loss, 'Validation': val_loss}, epoch)
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    writer.close()
    return training_losses,validation_losses

In [ ]:
# Train the Model
epochs=30
training_losses,validation_losses=train_model(model, training_data, validation_data)

In [ ]:
# Task 6: Visualize Training and Validation Loss
plt.figure(figsize=(10,5))
plt.plot(range(1, epochs+1), training_losses, label='Train Loss', marker='o')
plt.plot(range(1, epochs+1), validation_losses, label='Validation Loss', marker='s')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

The neural network for regression is trained with batch size=30 over 30 epochs with SGD(stochastic gradient descent) as an optimizer and learning rate=0.01.
Low value for learning rate is suitable because it is an hyperparameter can learn during the training phase and  larger values, sometimes may lead to convergence problems. As number of epochs are increasing the model is able to learn the patterns and predictions are becoming familiar, we can observe that from the plots where training and validation losses are decreasing as number of epochs are increasing.

In [ ]:
# Task 7: Save the Trained Model
state_dict = model.state_dict()
print(state_dict)
torch.save(state_dict, "model.pt") #.pt/.pth  extensions

save_model(model)



In [ ]:
# Task 8: Load the Trained Model and Evaluate
def load_model(model, path="regression_model.pth"):
    model.load_state_dict(torch.load(path))
    model.eval()
    print(f"Model loaded from {path}")
    return model

# Evaluation of the model by using loss function as mean square error
def evaluate_model(model, test_loader):
    test_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            predictions = model(x_batch)
            loss = loss_fn(predictions, y_batch)
            test_loss += loss.item()
    test_loss /= len(test_loader)
    print(f"Test MSE: {test_loss:.4f}")

# Load and evaluate model
eval_model = load_model(model)
evaluate_model(eval_model, testing_data)

In [ ]:
# Task 9: Scatter  Plot of the predictions
def plot_predictions(model, test_loader):
    actuals, predictions = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            preds = model(x_batch)
            actuals.append(y_batch.numpy())
            predictions.append(preds.numpy())
    actuals = np.vstack(actuals)
    predictions = np.vstack(predictions)

    for i in range(actuals.shape[1]):
        plt.figure()
        plt.scatter(actuals[:, i], predictions[:, i], alpha=0.6, label="Predictions")

        # Fitting a regression line
        m, b = np.polyfit(actuals[:, i], predictions[:, i], 1)
        plt.plot(actuals[:, i], m * actuals[:, i] + b, color='red', label="Regression Line")

        plt.xlabel("True Values")
        plt.ylabel("Predicted Values")
        plt.title(f"Scatter Plot with Prediction Line for y_{i+1}")
        plt.legend()
        plt.grid(True)
        plt.show()

# Load and evaluate model
#eval_model = load_model(model)
plot_predictions(eval_model, testing_data)
